In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
# Get the directory where the current notebook is located
current_dir = Path.cwd()

# Define the path relative to the notebook
# This works regardless of the operating system
data_path = current_dir.parent / "data" / "merged_covid_data.csv"

In [2]:
data = pd.read_csv(data_path)

In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 170140 entries, 0 to 170139
Data columns (total 30 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   date                                170140 non-null  str    
 1   location_key                        170140 non-null  str    
 2   new_confirmed                       170052 non-null  float64
 3   new_deceased                        169886 non-null  float64
 4   new_recovered                       14952 non-null   float64
 5   new_tested                          73354 non-null   float64
 6   cumulative_confirmed                170053 non-null  float64
 7   cumulative_deceased                 169888 non-null  float64
 8   cumulative_recovered                14048 non-null   float64
 9   cumulative_tested                   76225 non-null   float64
 10  school_closing                      169740 non-null  float64
 11  workplace_closing                   1

In [ ]:
# ── Folium Choropleth with Month-by-Month Time Slider ──────────────────
import json, math
import folium
from folium.plugins import TimeSliderChoropleth
import branca.colormap as cm
import numpy as np
from pathlib import Path

current_dir = Path.cwd()
geojson_path = current_dir.parent / "data" / "countries.geojson"

# 1. Filter to country-level and aggregate by month
country_data = data[
    (data["location_key"].str.len() == 2) &
    (~data["location_key"].str.contains("_"))
].copy()

country_data["date"] = pd.to_datetime(country_data["date"])
country_data["month"] = country_data["date"].dt.to_period("M")

monthly = (
    country_data
    .groupby(["location_key", "month"], as_index=False)["new_confirmed"]
    .sum(min_count=1)  # keeps NaN when all daily values are NaN
)
monthly["unix_ts"] = (
    monthly["month"].dt.to_timestamp("D", how="start").astype("int64") // 10**9
)

# 2. Log-scale colormap
cap_value = float(np.percentile(monthly["new_confirmed"].dropna().pipe(lambda s: s[s > 0]), 99))

def log_normalize(val, cap):
    if pd.isna(val) or val < 0:
        return None
    return min(math.log1p(float(val)) / math.log1p(cap), 1.0)

colormap = cm.linear.YlOrRd_09.scale(0, 1)
colormap.caption = "Monthly New Confirmed COVID-19 Cases (log scale)"

# 3. Load GeoJSON and inject top-level id field (required by TimeSliderChoropleth)
with open(geojson_path, "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

for feature in geojson_data["features"]:
    feature["id"] = feature["properties"]["ISO3166-1-Alpha-2"]

geojson_str = json.dumps(geojson_data)

# 4. Build styledict: country_id -> {unix_ts_str -> {color, opacity}}
MISSING = {"color": "#d3d3d3", "opacity": 0.4}
DATA_OPACITY = 0.7

all_months = monthly[["month", "unix_ts"]].drop_duplicates().sort_values("month")
monthly_indexed = monthly.set_index(["location_key", "unix_ts"])["new_confirmed"]
geo_countries = {f["id"] for f in geojson_data["features"] if len(f["id"]) == 2}

styledict = {}
for cid in geo_countries:
    ts_style = {}
    for _, row in all_months.iterrows():
        ts_str = str(int(row["unix_ts"]))
        try:
            val = monthly_indexed.loc[(cid, int(row["unix_ts"]))]
        except KeyError:
            val = float("nan")
        norm = log_normalize(val, cap_value)
        ts_style[ts_str] = MISSING if norm is None else {"color": colormap(norm), "opacity": DATA_OPACITY}
    styledict[cid] = ts_style

# 5. Render map
m = folium.Map(location=[20, 0], zoom_start=2, tiles="CartoDB positron", min_zoom=1, max_bounds=True)
first_ts = sorted(next(iter(styledict.values())).keys())[0]
TimeSliderChoropleth(data=geojson_str, styledict=styledict, init_timestamp=first_ts).add_to(m)
colormap.add_to(m)
m